# 🌐 dlt Sources, Resources & Transformers : Ingestion d'une API REST E-Commerce

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dlt-hub/dlt/blob/master/docs/education/dlt-fundamentals-course/lesson_2_dlt_sources_and_resources_create_first_dlt_pipeline.ipynb)

Dans la [Leçon 1](01_lesson_quick_start.ipynb), nous avons vu comment charger des dictionnaires Python dans DuckDB et observer l'aplatissement automatique des données.

Dans cette deuxième leçon, nous passons à la vitesse supérieure en connectant une **véritable API REST e-commerce** ([FakeStore API](https://fakestoreapi.com)) :

### 🎯 Objectifs de ce lab :
1. Découvrir le décorateur `@dlt.resource` pour encapsuler et streamer des flux de données depuis une API REST.
2. Comprendre le déballage automatique des listes d'objets (*Nested Tables*) : comment `dlt` transforme les listes de produits d'un panier en sous-tables relationnelles.
3. Regrouper plusieurs endpoints sous une même `@dlt.source` modulaire.
4. Utiliser les **Transformers** (`@dlt.transformer`) pour enrichir les flux à la volée (ex: récupérer le profil d'un client pour chaque panier).
5. Contrôler la profondeur de normalisation avec `max_table_nesting`.


---
## 📦 Installation des dépendances

Nous installons `dlt` avec le driver `duckdb`, ainsi que `requests` pour les appels HTTP vers l'API e-commerce.


In [ ]:
%%capture
!pip install "dlt[duckdb]" requests


---
## 🧩 1. Le décorateur `@dlt.resource` : streamer des données depuis une API

Dans un pipeline professionnel, on ne charge pas toutes les données en mémoire vive dans une simple variable Python. On utilise une **fonction génératrice** (`yield`) décorée par `@dlt.resource`.

### Pourquoi utiliser `@dlt.resource` ?
* **Streaming par paquets (*Chunks*)** : Les données sont traitées flux par flux, sans risquer de saturer la RAM (*Out-Of-Memory*).
* **Configuration déclarative** : On définit le nom de la table cible (`table_name`) et le mode d'écriture (`write_disposition="replace"`, `"append"`, etc.).
* **Composabilité** : Les ressources peuvent être combinées, filtrées ou branchées à des transformers.

Commençons par ingérer le catalogue de produits depuis l'endpoint `https://fakestoreapi.com/products` :


In [ ]:
import dlt
from dlt.sources.helpers import requests

@dlt.resource(table_name="products", write_disposition="replace")
def get_ecommerce_products():
    """Ressource dlt extrayant le catalogue de produits depuis l'API FakeStore."""
    url = "https://fakestoreapi.com/products"
    response = requests.get(url)
    response.raise_for_status()
    products = response.json()
    print(f"📦 {len(products)} produits récupérés depuis l'API.")
    yield products


---
### Exécution du pipeline pour charger les produits

Créons un pipeline `fakestore_pipeline` et lançons le chargement de notre ressource :


In [ ]:
# Initialisation du pipeline dlt
pipeline = dlt.pipeline(
    pipeline_name="fakestore_pipeline",
    destination="duckdb",
    dataset_name="fakestore_raw",
    dev_mode=True,
)

# Ingestion de la ressource 'products'
load_info = pipeline.run(get_ecommerce_products)
print(load_info)


Inspectons les colonnes générées.
Notez que chaque produit de l'API FakeStore possède un champ `rating: {"rate": 4.1, "count": 259}`.
`dlt` a automatiquement extrait ce dictionnaire imbriqué sous la forme de deux colonnes : `rating__rate` et `rating__count` !


In [ ]:
df_products = pipeline.dataset().products.df()
print("Colonnes générées dans la table products :\n", df_products.columns.tolist())

# Affichage d'un extrait des produits
df_products[['id', 'title', 'price', 'category', 'rating__rate', 'rating__count']].head()


---
## 🛒 2. Deuxième ressource : Les Paniers (`/carts`) et les listes imbriquées

Dans une boutique e-commerce, un panier d'achat contient une **liste d'articles commandés**.
Regardez la structure JSON renvoyée par `https://fakestoreapi.com/carts` :

```json
{
  "id": 1,
  "userId": 2,
  "date": "2020-03-02T00:00:00.000Z",
  "products": [
    {"productId": 1, "quantity": 4},
    {"productId": 5, "quantity": 1}
  ]
}
```

Définissons une seconde ressource pour extraire ces paniers :


In [ ]:
@dlt.resource(table_name="carts", write_disposition="replace")
def get_ecommerce_carts():
    """Ressource dlt extrayant les paniers d'achat contenant des listes d'articles."""
    url = "https://fakestoreapi.com/carts"
    response = requests.get(url)
    response.raise_for_status()
    carts = response.json()
    print(f"🛒 {len(carts)} paniers d'achat récupérés depuis l'API.")
    yield carts


---
### Charger plusieurs ressources dans le même pipeline

On peut passer une liste de plusieurs ressources directement à `pipeline.run([res1, res2])` :


In [ ]:
# Ingestion simultanée des produits et des paniers
load_info = pipeline.run([get_ecommerce_products, get_ecommerce_carts])
print(load_info)


---
### 💥 Le "Moment Clé" de dlt : Le déballage relationnel des listes (*Nested Tables*)

Interrogeons DuckDB pour voir les tables créées dans notre schéma. Vous allez constater une surprise :


In [ ]:
import duckdb

conn = duckdb.connect(f"{pipeline.pipeline_name}.duckdb")
conn.sql(f"SET search_path = '{pipeline.dataset_name}'")

print("Tables présentes dans la base :")
display(conn.sql("SHOW TABLES").df())


💡 **Observez la table `carts__products` !**
D'où vient cette table alors que nous ne l'avons jamais définie ?

- Dans une base SQL classique, on ne peut pas stocker proprement un tableau JSON sans perdre la puissance de requêtage relationnelle.
- **`dlt` a automatiquement déballé (*unpacked*) la liste `products` dans une table fille dédiée : `carts__products`**.
- La table fille contient les champs `product_id` et `quantity`, ainsi qu'une clé étrangère `_dlt_parent_id` pointant vers l'identifiant technique `_dlt_id` de la table parente `carts` !

C'est exactement la relation **`orders` ➔ `order_items`** que nous retrouvons dans le fil rouge TheLook !


In [ ]:
# Jointure SQL directe entre la table parente 'carts' et la table fille 'carts__products'
query = """
SELECT 
    c.id AS cart_id,
    c.user_id,
    c.date,
    cp.product_id,
    cp.quantity
FROM carts c
JOIN carts__products cp ON c._dlt_id = cp._dlt_parent_id
LIMIT 10
"""
conn.sql(query).df()


---
## 📦 3. Le décorateur `@dlt.source` : regrouper logiquement les ressources

En production, une application ou un domaine métier comprend de multiples endpoints (produits, paniers, utilisateurs, stocks).
Le décorateur `@dlt.source` permet de regrouper ces ressources dans un conteneur logique réutilisable :


In [ ]:
from typing import Iterable
from dlt.extract import DltResource

@dlt.source(name="fakestore_shop")
def fakestore_source() -> Iterable[DltResource]:
    """Source unifiée regroupant l'ensemble des flux du magasin e-commerce."""
    return get_ecommerce_products, get_ecommerce_carts

# Exécution de la source complète
load_info = pipeline.run(fakestore_source())
print(load_info)


---
## 🔄 4. Les Transformers (`@dlt.transformer`) : enrichir les flux à la volée

Imaginez le scénario suivant :
Pour chaque panier chargé, nous avons l'identifiant du client (`userId`), mais pas ses coordonnées (email, nom, adresse).
Nous voulons interroger l'API `/users/{userId}` pour enrichir la commande avec le profil du client au fur et à mesure que les paniers sont ingérés.

C'est le rôle des **Transformers** `dlt` : une ressource spéciale qui consomme les éléments émis par une ressource parent.


In [ ]:
@dlt.transformer(data_from=get_ecommerce_carts, table_name="cart_customers")
def get_customer_for_cart(cart):
    """Transformer qui récupère le profil complet du client pour chaque panier émis."""
    user_id = cart["userId"]
    user_url = f"https://fakestoreapi.com/users/{user_id}"
    response = requests.get(user_url)
    if response.status_code == 200:
        user_data = response.json()
        # On attache l'identifiant du panier pour la traçabilité
        user_data["cart_id"] = cart["id"]
        yield user_data


---
### L'opérateur Pipe (`|`) pour chaîner ressources et transformers

`dlt` permet de brancher un transformer à une ressource avec une syntaxe très élégante : `ressource | transformer` :


In [ ]:
# Enchaînement : les paniers sont envoyés dans le transformer
load_info = pipeline.run(get_ecommerce_carts | get_customer_for_cart)
print(load_info)


Inspectons la table `cart_customers` générée.
Remarquez comment `dlt` a aplati l'adresse du client et ses coordonnées GPS (`address__city`, `address__geolocation__lat`) sans aucun effort :


In [ ]:
df_customers = pipeline.dataset().cart_customers.df()
print("Colonnes générées dans cart_customers :\n", df_customers.columns.tolist())

df_customers[['cart_id', 'email', 'name__firstname', 'name__lastname', 'address__city', 'address__geolocation__lat']].head()


---
## ⚙️ 5. Contrôler le niveau de déballage (`max_table_nesting`)

Par défaut, `dlt` déballe toutes les listes imbriquées en sous-tables relationnelles.
Si vous préférez conserver le JSON brut dans une colonne sans créer de sous-tables (par exemple pour traiter le JSON plus tard dans BigQuery avec `JSON_EXTRACT` ou `UNNEST`), vous pouvez utiliser le paramètre `max_table_nesting=0` :


In [ ]:
# Ressource avec désactivation des sous-tables
@dlt.resource(table_name="carts_raw_json", max_table_nesting=0, write_disposition="replace")
def get_carts_without_subtables():
    url = "https://fakestoreapi.com/carts"
    response = requests.get(url)
    yield response.json()

pipeline.run(get_carts_without_subtables)

# Observation : le champ 'products' est conservé sous forme de chaîne JSON brute
df_raw = pipeline.dataset().carts_raw_json.df()
df_raw[['id', 'user_id', 'products']].head()


---
## 🏋️ Exercice d'application : Ingestion autonome de la ressource `users`

À vous de jouer !

### Consignes :
1. Créez une ressource `@dlt.resource(table_name="users", write_disposition="replace")` qui extrait tous les utilisateurs depuis l'endpoint `https://fakestoreapi.com/users`.
2. Exécutez le pipeline pour charger cette ressource dans DuckDB.
3. À l'aide de `conn.sql()` ou `pipeline.sql_client()`, écrivez une requête SQL pour afficher le **nombre de clients par ville (`address__city`)**.


In [ ]:
# 1. Définissez la ressource 'get_ecommerce_users'
@dlt.resource(table_name="users", write_disposition="replace")
def get_ecommerce_users():
    url = "https://fakestoreapi.com/users"
    response = requests.get(url)
    response.raise_for_status()
    yield response.json()

# 2. Exécutez le chargement
pipeline.run(get_ecommerce_users)

# 3. Requête SQL analytique : répartition des clients par ville
with pipeline.sql_client() as client:
    sql = """
    SELECT 
        address__city,
        COUNT(*) AS nb_clients
    FROM users
    GROUP BY address__city
    ORDER BY nb_clients DESC
    """
    with client.execute_query(sql) as cursor:
        display(cursor.df())


---
### 🎓 Synthèse du Jour 1 & Pont vers le Jour 2

Au cours de cette journée, vous avez construit la base solide du Data Engineering :
1. **Compréhension du paradigme Big Data & OLAP** (BigQuery, stockage colonne, découplage calcul/stockage).
2. **Mise en place d'un Data Lake & DWH** sur Cloud Storage et BigQuery.
3. **Ingestion automatisée avec `dlt`** : extraction d'APIs REST, inférence de schémas, aplatissement de JSON et déballage relationnel des listes.

👉 **Ce que nous ferons le Jour 2 :**
Ces tables brutes ingérées constituent la **couche Bronze** de votre architecture Lakehouse.
Demain, nous brancherons **`dbt` (data build tool)** directement sur BigQuery pour nettoyer, dédupliquer et modéliser ces données brutes vers les couches **Silver** et **Gold** !
